In [16]:
import os
import sys
from pathlib import Path
import psycopg2
from sshtunnel import SSHTunnelForwarder
import paramiko
from dotenv import load_dotenv
import pandas as pd

rootDir = Path(os.path.abspath(__name__)).parents[1]
sys.path.append(str(rootDir))

from src.db_client import Client

load_dotenv()

True

In [17]:
client = Client(local_port=6000)

In [22]:
qry = """
    SELECT 
        s.scientific_name,
        ce.specific_longitude as long,
        ce.specific_latitude as lat
    FROM specimen s
    JOIN collection_event ce USING(collection_id);"""

client.pull_data(qry)

,scientific_name,long,lat
0,Orthorhynchula linneyi ...,-84.382964,39.3338
1,Orthorhynchula linneyi ...,-84.382964,39.3338


In [2]:
SSH_HOST = os.getenv("SSH_HOST")
SSH_USERNAME = os.getenv("SSH_USERNAME")
SSH_PRIVATE_KEY = os.getenv("SSH_PRIVATE_KEY_PATH")
SSH_PASSWORD = os.getenv("SSH_PRIVATE_KEY_PASSWORD")

DATABASE = os.getenv("DATABASE")
USER = os.getenv("PG_USER")
DB_PASSWORD = os.getenv("PG_PASSWORD")

In [6]:
key = paramiko.Ed25519Key.from_private_key_file(SSH_PRIVATE_KEY, password=SSH_PASSWORD)

with SSHTunnelForwarder(
    ('192.168.50.186', 22),
    ssh_username=SSH_USERNAME,
    ssh_pkey=key,
    remote_bind_address=('localhost', 5432),
    local_bind_address=('127.0.0.1', 6000)) as tunnel:
    
    print(f"SSH tunnel established. Local port: {tunnel.local_bind_port}")

    conn = psycopg2.connect(
        host='localhost',
        port=tunnel.local_bind_port,
        database=DATABASE,
        user=USER,
        password=DB_PASSWORD
    )

    sql_query = "SELECT * FROM collection_event;"
    df = pd.read_sql_query(sql_query, conn)

SSH tunnel established. Local port: 6000


In [7]:
df

,collection_id,field_number,locality_id,collection_date,collector_id,participants,collection_remarks,created_at,formation,formation_code,specific_formation,stage,specific_latitude,specific_longitude,elevation
0,893ca33b-f1b7-4022-b86e-19b839509ef2,24-01,d96a44d3-da00-421a-b50b-975cfc0ab4b7,2024-06-16,65d630ee-51ff-464a-8370-777295a76f1e,"{""65d630ee-51ff-464a-8370-777295a76f1e"", ""5810...",Came to Keehner park to look for fossils/creat...,2025-01-05 10:18:28.301797,Grant Lake Formation,Oglf,Grant Lake Formation,Maysvillian Stage,39.3338,-84.382964,765.0
